# data_preparation_historical_risk_patch

Notebook hasil konversi dari `data_preparation_historical_risk_patch.py`. Kode dipisah ke beberapa cell berdasarkan blok top-level agar lebih mudah dibaca, dijalankan, dan di-screenshot.


## Imports


In [ ]:
from __future__ import annotations

import argparse
import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFilter


## Konstanta dan Setup Awal


In [ ]:
DATE_PATTERN = re.compile(r"FIRMS_(\d{4}-\d{2}-\d{2})")
DEFAULT_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


## Class: `DatasetRecord`


In [ ]:
class DatasetRecord:
    path: Path
    date: datetime


## Function: `parse_image_extensions`


In [ ]:
def parse_image_extensions(value: str) -> tuple[str, ...]:
    extensions: list[str] = []
    for item in value.split(","):
        ext = item.strip().lower()
        if not ext:
            continue
        if not ext.startswith("."):
            ext = f".{ext}"
        if ext not in extensions:
            extensions.append(ext)
    return tuple(extensions) if extensions else DEFAULT_IMAGE_EXTENSIONS


## Function: `load_records`


In [ ]:
def load_records(dataset_dir: Path, image_extensions: tuple[str, ...]) -> list[DatasetRecord]:
    records: list[DatasetRecord] = []
    allowed_extensions = set(image_extensions)
    for path in sorted(item for item in dataset_dir.iterdir() if item.is_file()):
        if path.suffix.lower() not in allowed_extensions:
            continue
        match = DATE_PATTERN.search(path.name)
        if not match:
            continue
        records.append(DatasetRecord(path=path, date=datetime.strptime(match.group(1), "%Y-%m-%d")))
    return sorted(records, key=lambda item: item.date)


## Function: `validate_records`


In [ ]:
def validate_records(records: list[DatasetRecord]) -> tuple[list[DatasetRecord], list[dict[str, str]]]:
    valid_records: list[DatasetRecord] = []
    skipped_records: list[dict[str, str]] = []
    for record in records:
        try:
            if not record.path.exists():
                raise FileNotFoundError(f"File tidak ditemukan: {record.path}")
            with Image.open(record.path) as image:
                image.verify()
            valid_records.append(record)
        except Exception as exc:
            skipped_records.append({"path": str(record.path), "reason": str(exc)})
    return valid_records, skipped_records


## Function: `build_sample_starts`


In [ ]:
def build_sample_starts(record_count: int, seq_length: int) -> list[int]:
    return list(range(record_count - seq_length))


## Function: `split_sample_starts`


In [ ]:
def split_sample_starts(
    sample_starts: list[int],
    train_ratio: float,
    val_ratio: float,
) -> tuple[list[int], list[int], list[int]]:
    sample_count = len(sample_starts)
    train_end = max(1, int(sample_count * train_ratio))
    val_end = max(train_end + 1, int(sample_count * (train_ratio + val_ratio)))
    val_end = min(val_end, sample_count - 1)
    train = sample_starts[:train_end]
    val = sample_starts[train_end:val_end]
    test = sample_starts[val_end:]
    if not val or not test:
        raise ValueError("Split train/val/test gagal. Periksa rasio dataset.")
    return train, val, test


## Function: `_normalize_kernel`


In [ ]:
def _normalize_kernel(size: int) -> int:
    size = max(1, int(size))
    return size if size % 2 == 1 else size + 1


## Function: `load_native_mask`


In [ ]:
def load_native_mask(
    path_str: str,
    native_width: int,
    native_height: int,
    dilation_kernel: int,
) -> np.ndarray:
    dilation_kernel = _normalize_kernel(dilation_kernel)
    with Image.open(path_str) as image:
        rgb = image.convert("RGB")
        hsv = np.asarray(rgb.convert("HSV"), dtype=np.uint8)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    red_low = (h <= 14) & (s >= 70) & (v >= 50)
    red_high = (h >= 242) & (s >= 70) & (v >= 50)
    mask = ((red_low | red_high).astype(np.uint8)) * 255

    mask_image = Image.fromarray(mask)
    if dilation_kernel > 1:
        mask_image = mask_image.filter(ImageFilter.MaxFilter(size=dilation_kernel))
    if mask_image.size != (native_width, native_height):
        mask_image = mask_image.resize((native_width, native_height), Image.BILINEAR)

    density = np.asarray(mask_image, dtype=np.float32) / 255.0
    return np.clip(density, 0.0, 1.0)


## Function: `load_native_risk_map`


In [ ]:
def load_native_risk_map(
    path_str: str,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
) -> np.ndarray:
    label_dilation_kernel = _normalize_kernel(label_dilation_kernel)
    with Image.open(path_str) as image:
        rgb = image.convert("RGB")
        hsv = np.asarray(rgb.convert("HSV"), dtype=np.uint8)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    red_low = (h <= 14) & (s >= 70) & (v >= 50)
    red_high = (h >= 242) & (s >= 70) & (v >= 50)
    mask = ((red_low | red_high).astype(np.uint8)) * 255

    risk_image = Image.fromarray(mask)
    if label_dilation_kernel > 1:
        risk_image = risk_image.filter(ImageFilter.MaxFilter(size=label_dilation_kernel))
    if label_blur_radius > 0:
        risk_image = risk_image.filter(ImageFilter.GaussianBlur(radius=float(label_blur_radius)))
    if risk_image.size != (native_width, native_height):
        risk_image = risk_image.resize((native_width, native_height), Image.BILINEAR)

    density = np.asarray(risk_image, dtype=np.float32) / 255.0
    return np.clip(density, 0.0, 1.0)


## Function: `sample_patch_centers`


In [ ]:
def sample_patch_centers(
    target_mask: np.ndarray,
    positive_patch_count: int,
    negative_patch_count: int,
    ground_truth_threshold: float,
    rng: np.random.Generator,
) -> list[tuple[int, int]]:
    binary = target_mask >= ground_truth_threshold
    positive_coords = np.argwhere(binary)
    centers: list[tuple[int, int]] = []

    if len(positive_coords) > 0 and positive_patch_count > 0:
        replace = len(positive_coords) < positive_patch_count
        indices = rng.choice(len(positive_coords), size=positive_patch_count, replace=replace)
        for idx in np.atleast_1d(indices):
            cy, cx = positive_coords[int(idx)]
            centers.append((int(cy), int(cx)))

    neg_needed = negative_patch_count
    attempts = max(neg_needed * 40, 40)
    while neg_needed > 0 and attempts > 0:
        cy = int(rng.integers(0, target_mask.shape[0]))
        cx = int(rng.integers(0, target_mask.shape[1]))
        if not binary[cy, cx]:
            centers.append((cy, cx))
            neg_needed -= 1
        attempts -= 1

    if not centers:
        centers.append((target_mask.shape[0] // 2, target_mask.shape[1] // 2))
    return centers


## Function: `extract_patch`


In [ ]:
def extract_patch(array: np.ndarray, cy: int, cx: int, patch_height: int, patch_width: int) -> np.ndarray:
    if array.ndim == 2:
        array = array[..., None]

    height, width = array.shape[:2]
    half_h = patch_height // 2
    half_w = patch_width // 2
    top = cy - half_h
    left = cx - half_w
    bottom = top + patch_height
    right = left + patch_width

    pad_top = max(0, -top)
    pad_left = max(0, -left)
    pad_bottom = max(0, bottom - height)
    pad_right = max(0, right - width)

    if pad_top or pad_left or pad_bottom or pad_right:
        array = np.pad(array, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="constant")
        top += pad_top
        left += pad_left
        bottom = top + patch_height
        right = left + patch_width

    return array[top:bottom, left:right, :].astype(np.float32)


## Function: `build_patch_entries`


In [ ]:
def build_patch_entries(
    records: list[DatasetRecord],
    sample_starts: list[int],
    *,
    seq_length: int,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
    ground_truth_threshold: float,
    positive_patches: int,
    negative_patches: int,
    seed: int,
) -> tuple[list[tuple[int, int, int]], dict[str, int]]:
    rng = np.random.default_rng(seed)
    entries: list[tuple[int, int, int]] = []
    positive_samples = 0
    negative_samples = 0

    for start in sample_starts:
        target_mask = load_native_risk_map(
            str(records[start + seq_length].path),
            native_width,
            native_height,
            label_dilation_kernel,
            label_blur_radius,
        )
        has_positive = bool(np.any(target_mask >= ground_truth_threshold))
        if has_positive:
            positive_samples += 1
        else:
            negative_samples += 1

        centers = sample_patch_centers(
            target_mask,
            positive_patch_count=positive_patches if has_positive else 0,
            negative_patch_count=negative_patches,
            ground_truth_threshold=ground_truth_threshold,
            rng=rng,
        )
        entries.extend((start, cy, cx) for cy, cx in centers)

    stats = {
        "sample_count": len(sample_starts),
        "positive_sample_count": positive_samples,
        "negative_sample_count": negative_samples,
        "patch_entry_count": len(entries),
        "positive_patches_per_sample": positive_patches,
        "negative_patches_per_sample": negative_patches,
    }
    return entries, stats


## Function: `summarize_patch_entries`


In [ ]:
def summarize_patch_entries(
    records: list[DatasetRecord],
    sample_starts: list[int],
    *,
    seq_length: int,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
    ground_truth_threshold: float,
    positive_patches: int,
    negative_patches: int,
) -> dict[str, int]:
    positive_samples = 0
    negative_samples = 0
    patch_entry_count = 0

    for start in sample_starts:
        target_mask = load_native_risk_map(
            str(records[start + seq_length].path),
            native_width,
            native_height,
            label_dilation_kernel,
            label_blur_radius,
        )
        has_positive = bool(np.any(target_mask >= ground_truth_threshold))
        if has_positive:
            positive_samples += 1
            patch_entry_count += positive_patches + negative_patches
        else:
            negative_samples += 1
            patch_entry_count += negative_patches

    return {
        "sample_count": len(sample_starts),
        "positive_sample_count": positive_samples,
        "negative_sample_count": negative_samples,
        "patch_entry_count": patch_entry_count,
        "positive_patches_per_sample": positive_patches,
        "negative_patches_per_sample": negative_patches,
    }


## Function: `find_demo_sample_start`


In [ ]:
def find_demo_sample_start(
    records: list[DatasetRecord],
    train_starts: list[int],
    seq_length: int,
    input_dilation_kernel: int,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
    ground_truth_threshold: float,
    sample_mode: str,
) -> int:
    if sample_mode == "first_train":
        return train_starts[0]

    for start in train_starts:
        input_mask = load_native_mask(
            str(records[start + seq_length - 1].path),
            native_width,
            native_height,
            input_dilation_kernel,
        )
        target_mask = load_native_risk_map(
            str(records[start + seq_length].path),
            native_width,
            native_height,
            label_dilation_kernel,
            label_blur_radius,
        )
        if np.any(input_mask > 0) and np.any(target_mask >= ground_truth_threshold):
            return start

    for start in train_starts:
        target_mask = load_native_risk_map(
            str(records[start + seq_length].path),
            native_width,
            native_height,
            label_dilation_kernel,
            label_blur_radius,
        )
        if np.any(target_mask >= ground_truth_threshold):
            return start
    return train_starts[0]


## Function: `array_to_image`


In [ ]:
def array_to_image(array: np.ndarray) -> Image.Image:
    array_2d = array[..., 0] if array.ndim == 3 else array
    array_uint8 = np.clip(array_2d * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(array_uint8)


## Function: `build_centers_overlay`


In [ ]:
def build_centers_overlay(base_map: np.ndarray, centers: list[tuple[int, int]], radius: int = 5) -> Image.Image:
    image = array_to_image(base_map).convert("RGB")
    draw = ImageDraw.Draw(image)
    for cy, cx in centers:
        draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), outline=(255, 0, 0), width=2)
    return image


## Function: `save_preview_images`


In [ ]:
def save_preview_images(
    output_dir: Path,
    input_record: DatasetRecord,
    target_record: DatasetRecord,
    input_mask: np.ndarray,
    risk_map: np.ndarray,
    centers: list[tuple[int, int]],
    mask_patch: np.ndarray,
    risk_patch: np.ndarray,
) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_files: list[str] = []

    with Image.open(input_record.path) as image:
        input_original = image.convert("RGB")
        input_original_path = output_dir / "01_input_original.png"
        input_original.save(input_original_path)
        saved_files.append(str(input_original_path))

    with Image.open(target_record.path) as image:
        original = image.convert("RGB")
        original_path = output_dir / "02_target_original.png"
        original.save(original_path)
        saved_files.append(str(original_path))

    mask_path = output_dir / "03_input_mask.png"
    array_to_image(input_mask).save(mask_path)
    saved_files.append(str(mask_path))

    risk_path = output_dir / "04_risk_map.png"
    array_to_image(risk_map).save(risk_path)
    saved_files.append(str(risk_path))

    overlay_path = output_dir / "05_risk_map_with_patch_centers.png"
    build_centers_overlay(risk_map, centers).save(overlay_path)
    saved_files.append(str(overlay_path))

    mask_patch_path = output_dir / "06_example_input_patch.png"
    array_to_image(mask_patch).save(mask_patch_path)
    saved_files.append(str(mask_patch_path))

    risk_patch_path = output_dir / "07_example_target_patch.png"
    array_to_image(risk_patch).save(risk_patch_path)
    saved_files.append(str(risk_patch_path))

    return saved_files


## Function: `build_parser`


In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Demo Data Preparation historical risk patch. "
            "Script ini hanya menjalankan preprocessing inti tanpa training model."
        )
    )
    parser.add_argument(
        "--dataset-dir",
        default="Ipynb/Dataset History Fire Hotspot In Riau Province PNG",
        help="Folder dataset gambar hotspot historis.",
    )
    parser.add_argument("--image-extensions", default=".png")
    parser.add_argument("--seq-length", type=int, default=7)
    parser.add_argument("--train-ratio", type=float, default=0.70)
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument("--native-width", type=int, default=1528)
    parser.add_argument("--native-height", type=int, default=773)
    parser.add_argument("--input-dilation-kernel", type=int, default=5)
    parser.add_argument("--label-dilation-kernel", type=int, default=9)
    parser.add_argument("--label-blur-radius", type=float, default=2.0)
    parser.add_argument("--ground-truth-threshold", type=float, default=0.05)
    parser.add_argument("--positive-patches", type=int, default=4)
    parser.add_argument("--negative-patches", type=int, default=1)
    parser.add_argument("--patch-width", type=int, default=160)
    parser.add_argument("--patch-height", type=int, default=160)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--sample-mode",
        choices=["auto_positive", "first_train"],
        default="auto_positive",
        help="Pilih sample demo pertama yang mengandung hotspot atau sample train pertama.",
    )
    parser.add_argument(
        "--with-patch-entry-stats",
        action="store_true",
        help="Hitung statistik patch entries seperti pipeline training utama.",
    )
    parser.add_argument(
        "--patch-entry-scope",
        choices=["train", "val", "test", "all"],
        default="train",
        help="Pilih split yang dihitung saat --with-patch-entry-stats dipakai.",
    )
    parser.add_argument(
        "--save-preview-dir",
        default="",
        help="Simpan preview preprocessing ke folder ini. Kosongkan jika tidak perlu.",
    )
    parser.add_argument("--json", action="store_true")
    return parser


## Function: `build_summary`


In [ ]:
def build_summary(args: argparse.Namespace) -> dict:
    dataset_dir = Path(args.dataset_dir).expanduser().resolve()
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {dataset_dir}")

    image_extensions = parse_image_extensions(args.image_extensions)
    records = load_records(dataset_dir, image_extensions)
    valid_records, skipped_records = validate_records(records)
    if len(valid_records) <= args.seq_length:
        raise ValueError("Jumlah data valid tidak cukup untuk membentuk sequence.")

    sample_starts = build_sample_starts(len(valid_records), args.seq_length)
    train_starts, val_starts, test_starts = split_sample_starts(sample_starts, args.train_ratio, args.val_ratio)

    demo_start = find_demo_sample_start(
        valid_records,
        train_starts,
        args.seq_length,
        args.input_dilation_kernel,
        args.native_width,
        args.native_height,
        args.label_dilation_kernel,
        args.label_blur_radius,
        args.ground_truth_threshold,
        args.sample_mode,
    )
    target_record = valid_records[demo_start + args.seq_length]
    input_record = valid_records[demo_start + args.seq_length - 1]

    input_mask = load_native_mask(
        str(input_record.path),
        args.native_width,
        args.native_height,
        args.input_dilation_kernel,
    )
    risk_map = load_native_risk_map(
        str(target_record.path),
        args.native_width,
        args.native_height,
        args.label_dilation_kernel,
        args.label_blur_radius,
    )

    rng = np.random.default_rng(args.seed)
    centers = sample_patch_centers(
        risk_map,
        positive_patch_count=args.positive_patches,
        negative_patch_count=args.negative_patches,
        ground_truth_threshold=args.ground_truth_threshold,
        rng=rng,
    )
    first_cy, first_cx = centers[0]
    mask_patch = extract_patch(input_mask, first_cy, first_cx, args.patch_height, args.patch_width)
    risk_patch = extract_patch(risk_map, first_cy, first_cx, args.patch_height, args.patch_width)

    summary: dict = {
        "dataset_dir": str(dataset_dir),
        "record_count_total": len(records),
        "record_count_valid": len(valid_records),
        "record_count_skipped": len(skipped_records),
        "train_samples": len(train_starts),
        "val_samples": len(val_starts),
        "test_samples": len(test_starts),
        "demo_sample_start_index": demo_start,
        "demo_input_date": input_record.date.date().isoformat(),
        "demo_target_date": target_record.date.date().isoformat(),
        "input_mask_shape": list(input_mask.shape),
        "risk_map_shape": list(risk_map.shape),
        "input_mask_positive_pixels": int(np.count_nonzero(input_mask > 0)),
        "risk_map_positive_pixels_thresholded": int(np.count_nonzero(risk_map >= args.ground_truth_threshold)),
        "sampled_patch_center_count": len(centers),
        "sampled_patch_centers_preview": [[int(cy), int(cx)] for cy, cx in centers[:10]],
        "first_input_patch_shape": list(mask_patch.shape),
        "first_target_patch_shape": list(risk_patch.shape),
        "input_record": asdict(input_record) | {"path": str(input_record.path)},
        "target_record": asdict(target_record) | {"path": str(target_record.path)},
        "patch_settings": {
            "patch_width": args.patch_width,
            "patch_height": args.patch_height,
            "positive_patches": args.positive_patches,
            "negative_patches": args.negative_patches,
            "ground_truth_threshold": args.ground_truth_threshold,
        },
        "preprocessing_settings": {
            "input_dilation_kernel": args.input_dilation_kernel,
            "label_dilation_kernel": args.label_dilation_kernel,
            "label_blur_radius": args.label_blur_radius,
        },
        "patch_entry_stats": None,
        "saved_preview_files": [],
    }

    if args.with_patch_entry_stats:
        patch_stats: dict[str, dict[str, int]] = {}
        if args.patch_entry_scope in {"train", "all"}:
            patch_stats["train"] = summarize_patch_entries(
                valid_records,
                train_starts,
                seq_length=args.seq_length,
                native_width=args.native_width,
                native_height=args.native_height,
                label_dilation_kernel=args.label_dilation_kernel,
                label_blur_radius=args.label_blur_radius,
                ground_truth_threshold=args.ground_truth_threshold,
                positive_patches=args.positive_patches,
                negative_patches=args.negative_patches,
            )
        if args.patch_entry_scope in {"val", "all"}:
            patch_stats["val"] = summarize_patch_entries(
                valid_records,
                val_starts,
                seq_length=args.seq_length,
                native_width=args.native_width,
                native_height=args.native_height,
                label_dilation_kernel=args.label_dilation_kernel,
                label_blur_radius=args.label_blur_radius,
                ground_truth_threshold=args.ground_truth_threshold,
                positive_patches=1,
                negative_patches=1,
            )
        if args.patch_entry_scope in {"test", "all"}:
            patch_stats["test"] = summarize_patch_entries(
                valid_records,
                test_starts,
                seq_length=args.seq_length,
                native_width=args.native_width,
                native_height=args.native_height,
                label_dilation_kernel=args.label_dilation_kernel,
                label_blur_radius=args.label_blur_radius,
                ground_truth_threshold=args.ground_truth_threshold,
                positive_patches=1,
                negative_patches=1,
            )
        summary["patch_entry_stats"] = patch_stats

    if args.save_preview_dir:
        saved_files = save_preview_images(
            output_dir=Path(args.save_preview_dir).expanduser().resolve(),
            input_record=input_record,
            target_record=target_record,
            input_mask=input_mask,
            risk_map=risk_map,
            centers=centers,
            mask_patch=mask_patch,
            risk_patch=risk_patch,
        )
        summary["saved_preview_files"] = saved_files

    return summary


## Function: `print_human_summary`


In [ ]:
def print_human_summary(summary: dict) -> None:
    print("[data_preparation] Dataset valid:", summary["record_count_valid"])
    print(
        "[data_preparation] Demo sample:",
        f"input {summary['demo_input_date']} -> target {summary['demo_target_date']}",
    )
    print("[data_preparation] Input mask shape:", tuple(summary["input_mask_shape"]))
    print("[data_preparation] Risk map shape:", tuple(summary["risk_map_shape"]))
    print("[data_preparation] Input mask positive pixels:", summary["input_mask_positive_pixels"])
    print(
        "[data_preparation] Risk map positive pixels (thresholded):",
        summary["risk_map_positive_pixels_thresholded"],
    )
    print("[data_preparation] Sampled patch centers:", summary["sampled_patch_center_count"])
    print("[data_preparation] Centers preview:", summary["sampled_patch_centers_preview"])
    print("[data_preparation] Example input patch shape:", tuple(summary["first_input_patch_shape"]))
    print("[data_preparation] Example target patch shape:", tuple(summary["first_target_patch_shape"]))

    patch_stats = summary["patch_entry_stats"]
    if patch_stats is not None:
        if "train" in patch_stats:
            print("[data_preparation] Train patch entries:", patch_stats["train"]["patch_entry_count"])
        if "val" in patch_stats:
            print("[data_preparation] Val patch entries:", patch_stats["val"]["patch_entry_count"])
        if "test" in patch_stats:
            print("[data_preparation] Test patch entries:", patch_stats["test"]["patch_entry_count"])
    else:
        print("[data_preparation] Patch entry stats: dilewati")
        print("[data_preparation] Tips: pakai --with-patch-entry-stats untuk menghitung statistik penuh.")

    if summary["saved_preview_files"]:
        print("[data_preparation] Preview files:")
        for path in summary["saved_preview_files"]:
            print("-", path)


## Function: `main`


In [ ]:
def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    summary = build_summary(args)
    print_human_summary(summary)
    if args.json:
        print()
        print(json.dumps(summary, indent=2, default=str))


## Entry Point


In [ ]:
if __name__ == "__main__":
    main()
